In [6]:
import pandas as pd
import numpy as np
import os
import pyarrow
import fastparquet

ModuleNotFoundError: No module named 'pyarrow'

## Pickle — Python Binary Serialization

Python's built-in `pickle` module serializes any Python object to binary.

- `df.to_pickle(path)` — write a DataFrame to disk in pickle format.
- `pd.read_pickle(path)` — read it back.

### Limitations

- Pickle files are **Python-only** — not readable by other languages.
- Format stability is **not guaranteed** across library versions — use only for short-term storage.
- pandas tries to maintain backward compatibility, but pickle format may break across major versions.

### Other Open Source Binary Formats

pandas also supports:

| Format | Function | Notes |
|--------|----------|-------|
| Apache Parquet | `pd.read_parquet` | Requires `pyarrow` (`conda install pyarrow`) |
| HDF5 | `pd.HDFStore`, `pd.read_hdf` | Requires `pytables` (`conda install pytables`) |
| ORC | `pd.read_orc` | Requires `pyarrow` |
| Feather | `pd.read_feather` | Requires `pyarrow` |

These open formats embed type information — no type inference needed at read time.

## Key Points

- `to_pickle` / `pd.read_pickle` are the quickest way to persist a pandas object.
- Pickle is Python-only and not suitable for long-term or cross-language storage.
- Prefer open formats (Parquet, HDF5) for durability, interoperability, and performance.

In [3]:
frame = pd.read_csv("../examples/ex1.csv")

frame

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [4]:
frame.to_pickle("../examples/frame_pickle")

pd.read_pickle("../examples/frame_pickle")

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [5]:
# Apache Parquet — fast, compressed, cross-language binary format
fec = pd.read_parquet("../datasets/fec/fec.parquet")

fec.head()

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

## Reading Microsoft Excel Files

pandas reads Excel files via `pd.ExcelFile` or `pd.read_excel`.

- Requires `openpyxl` (for `.xlsx`) and `xlrd` (for old `.xls`): `conda install openpyxl xlrd`.

### `pd.ExcelFile` — When Reading Multiple Sheets

Creating an `ExcelFile` object opens the file once, making it efficient to read multiple sheets:

```python
xlsx = pd.ExcelFile("file.xlsx")
xlsx.sheet_names          # list available sheets
xlsx.parse("Sheet1")      # read a sheet into a DataFrame
xlsx.parse("Sheet1", index_col=0)  # set index column
```

### `pd.read_excel` — Single Sheet Shortcut

Pass the file path directly without creating an `ExcelFile` object:

```python
pd.read_excel("file.xlsx", sheet_name="Sheet1")
```

### Writing to Excel

Two approaches:

```python
# Via ExcelWriter (multiple sheets)
writer = pd.ExcelWriter("out.xlsx")
df.to_excel(writer, "Sheet1")
writer.save()

# Direct path (single sheet)
df.to_excel("out.xlsx")
```

## Key Points

- Use `pd.ExcelFile` when reading multiple sheets from the same file — opens the file only once.
- `pd.read_excel` is the quick single-sheet shortcut.
- `index_col=` works the same as in `read_csv`.
- `ExcelWriter` allows writing multiple sheets; `to_excel(path)` is the single-sheet shortcut.

In [ ]:
xlsx = pd.ExcelFile("../examples/ex1.xlsx")
xlsx.sheet_names

In [ ]:
xlsx.parse(sheet_name="Sheet1")

In [ ]:
xlsx.parse(sheet_name="Sheet1", index_col=0)

In [ ]:
frame = pd.read_excel("../examples/ex1.xlsx", sheet_name="Sheet1")

frame

In [ ]:
# Write to Excel — direct path shortcut
frame.to_excel("../examples/ex2.xlsx")

## HDF5 Format

HDF5 (Hierarchical Data Format) is designed for storing large quantities of scientific array data.

- Stores **multiple datasets and metadata** in a single file.
- Supports **on-the-fly compression** — repeated patterns stored efficiently.
- Enables reading/writing **small sections of large arrays** without loading everything into memory.
- Cross-language: also available in Java, Julia, MATLAB, etc.
- Requires `pytables`: `conda install pytables` (PyPI name: `tables`).

### `pd.HDFStore` — Dictionary-Like Interface

```python
store = pd.HDFStore("file.h5")
store["key"] = df          # write (fixed format)
store["key"]               # read back
store.put("key", df, format="table")       # write in table format
store.select("key", where=["index >= 10"]) # query (table format only)
store.close()
```

### Storage Schemas

| Schema | Speed | Supports Queries |
|--------|-------|------------------|
| `"fixed"` (default) | Faster | No |
| `"table"` | Slower | Yes — via `store.select(where=...)` |

### Shortcuts

- `df.to_hdf(path, key, format="table")` — write without managing `HDFStore` directly.
- `pd.read_hdf(path, key, where=[...])` — read with optional query filter.

### When to Use HDF5

- Best for **write-once, read-many** local datasets too large for memory.
- **Not a database** — concurrent writes from multiple writers can corrupt the file.
- For remote/distributed storage (S3, HDFS), prefer Apache Parquet.

## Key Points

- `HDFStore` works like a dict: assign to write, index to read.
- `format="table"` is slower to write but enables `select` queries on subsets of rows.
- `to_hdf` / `pd.read_hdf` are convenient shortcuts for single-object workflows.
- HDF5 is ideal for large local datasets; not suited for concurrent writes.

In [ ]:
frame = pd.DataFrame({"a": np.random.standard_normal(100)})

store = pd.HDFStore("../examples/mydata.h5")
store["obj1"] = frame
store["obj1_col"] = frame["a"]

store

In [ ]:
store["obj1"]

In [ ]:
# Table format enables query operations
store.put("obj2", frame, format="table")

store.select("obj2", where=["index >= 10 and index <= 15"])

In [ ]:
store.close()

In [ ]:
# Shortcut: to_hdf / read_hdf
frame.to_hdf("../examples/mydata.h5", "obj3", format="table")

pd.read_hdf("../examples/mydata.h5", "obj3", where=["index < 5"])

In [ ]:
os.remove("../examples/mydata.h5")